# XGBoost

Gradient boosted decision trees with L1/L2 regularization. State-of-the-art for tabular data — fast training, handles missing values natively, and supports GPU acceleration.

**When to Use:**
- Tabular data with complex non-linear patterns
- Competitions and production ML where accuracy matters
- Handles missing values natively (no imputation needed)
- When you need fine control over regularization and boosting

**Key Assumptions / Considerations:**
- Sufficient data for the ensemble to learn patterns (not ideal for very small datasets)
- Hyperparameter tuning is important (learning_rate, max_depth, regularization)
- Can overfit on noisy data without proper regularization
- Early stopping recommended to avoid overfitting
- Feature importance available in 3 types: weight (frequency), gain, and cover

**References:**
- [XGBoost Documentation](https://xgboost.readthedocs.io/en/stable/)
- [XGBoost sklearn API](https://xgboost.readthedocs.io/en/stable/python/python_api.html#module-xgboost.sklearn)
- [XGBoost Parameters](https://xgboost.readthedocs.io/en/stable/parameter.html)

In [ ]:
# pip install xgboost

import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve
)

from xgboost import XGBRegressor, XGBClassifier

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# test data 

np.random.seed(100)
n_samples = 8000

# numeric
age = np.random.randint(18, 70, n_samples)
income = np.random.normal(50000, 15000, n_samples)
num_transactions = np.random.poisson(10, n_samples)

# categorical
gender = np.random.choice(['Male', 'Female'], n_samples)
region = np.random.choice(['North', 'South', 'East', 'West'], n_samples)
product_type = np.random.choice(['A', 'B', 'C'], n_samples)

# continuous target
target_continuous = 0.05 * age + 0.0005 * income + 0.3 * num_transactions + np.random.normal(0, 2, n_samples)

# binary classification target
target_binary = (target_continuous > np.median(target_continuous)).astype(int)

df = pd.DataFrame({
    'age': age,
    'income': income,
    'num_transactions': num_transactions,
    'gender': gender,
    'region': region,
    'product_type': product_type,
    'target': target_continuous  # switch to target_binary for classification 
})

In [ ]:
# Load Data

#df = pd.read_csv("data.csv")

X = df.drop("target", axis=1)
y = df["target"]

In [ ]:
# Distribution

plt.figure(figsize=(8, 5))
sns.histplot(y, kde=True, bins=30)
plt.title("Target Variable Distribution")
plt.show()

print("Skewness:", round(y.skew(),4))
print("Kurtosis:", round(y.kurt(), 4))

In [ ]:
# Train/Test Split

try:
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
except ValueError:
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

In [ ]:
# Preprocessing

numeric_features = ['age', 'income', 'num_transactions']
categorical_features = ['gender', 'region', 'product_type']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [ ]:
# Parameter Grids

models = {
    "XGB_Regressor": (XGBRegressor(random_state=42, n_jobs=-1, verbosity=0), {
        "clf__n_estimators": [100, 300, 500],
        "clf__max_depth": [3, 5, 7],
        "clf__learning_rate": [0.01, 0.05, 0.1],
        "clf__subsample": [0.8, 1.0],
        "clf__colsample_bytree": [0.8, 1.0],
        "clf__reg_alpha": [0, 0.1, 1.0],
        "clf__reg_lambda": [1.0, 5.0],
    }),
    "XGB_Classifier": (XGBClassifier(random_state=42, n_jobs=-1, verbosity=0, eval_metric="logloss"), {
        "clf__n_estimators": [100, 300, 500],
        "clf__max_depth": [3, 5, 7],
        "clf__learning_rate": [0.01, 0.05, 0.1],
        "clf__subsample": [0.8, 1.0],
        "clf__colsample_bytree": [0.8, 1.0],
        "clf__reg_alpha": [0, 0.1, 1.0],
        "clf__reg_lambda": [1.0, 5.0],
    }),
}

# Cross Validation

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1) if len(y.unique()) < 10 else KFold(n_splits=5, shuffle=True, random_state=1)

In [ ]:
# Training

def train_models(X_train, y_train, X_test, y_test):

    results = []
    fitted_pipelines = []

    for name, (model, params) in models.items():
        print(f"\n\U0001f539 Training {name}...")

        is_classifier = isinstance(model, XGBClassifier)

        pipe = Pipeline([("prep", preprocessor), ("clf", model)])

        scoring = "roc_auc" if is_classifier else "r2"

        grid = GridSearchCV(pipe, param_grid=params, cv=cv, n_jobs=-1, scoring=scoring, return_train_score=False)

        try:
            y_train_use = y_train if not is_classifier else (y_train > np.median(y_train)).astype(int)
            y_test_use = y_test if not is_classifier else (y_test > np.median(y_test)).astype(int)

            grid.fit(X_train, y_train_use)
            best_model = grid.best_estimator_
            fitted_pipelines.append((name, best_model))
            y_pred = best_model.predict(X_test)

            if is_classifier:
                y_proba = best_model.predict_proba(X_test)[:, 1]
                metrics = {
                    "Model": name,
                    "Best Params": grid.best_params_,
                    "Accuracy": accuracy_score(y_test_use, y_pred),
                    "Recall": recall_score(y_test_use, y_pred),
                    "Precision": precision_score(y_test_use, y_pred),
                    "F1 Score": f1_score(y_test_use, y_pred),
                    "ROC-AUC": roc_auc_score(y_test_use, y_proba),
                }
            else:
                n = len(y_test_use)
                p = X_train.shape[1]
                rss = np.sum((y_test_use - y_pred) ** 2)
                mse = mean_squared_error(y_test_use, y_pred)
                aic = n * np.log(rss / n) + 2 * p
                bic = n * np.log(rss / n) + p * np.log(n)
                metrics = {
                    "Model": name,
                    "Best Params": grid.best_params_,
                    "RMSE": np.sqrt(mse),
                    "MAE": mean_absolute_error(y_test_use, y_pred),
                    "R\u00b2 Score": r2_score(y_test_use, y_pred),
                    "AIC": aic,
                    "BIC": bic,
                }

            results.append(metrics)
        except Exception as e:
            print(f"\u26a0\ufe0f Skipping {name} due to error: {e}")
            continue

    return results, fitted_pipelines

In [ ]:
results, pipelines = train_models(X_train_full, y_train_full, X_test, y_test)

In [ ]:
# Results Summary

results_df = pd.DataFrame(results)
print("\n\u2705 Model Evaluation Summary:")
results_df

In [ ]:
# Best Model
regression_results = [r for r in results if 'R\u00b2 Score' in r]
if regression_results:
    best_model_name = max(regression_results, key=lambda x: x['R\u00b2 Score'])['Model']
else:
    best_model_name = max(results, key=lambda x: x.get('ROC-AUC', 0))['Model']

best_model_pipeline = [p for n, p in pipelines if n == best_model_name][0]
print(f"\n\U0001f3c6 Best Model: {best_model_name}")

In [ ]:
# Feature Importance

for name, pipe in pipelines:
    print(f"\n\U0001f4ca {name} Feature Importance:")
    
    xgb_model = pipe.named_steps["clf"]
    prep = pipe.named_steps["prep"]
    feature_names = prep.get_feature_names_out()
    
    # Default importance (gain)
    importances = xgb_model.feature_importances_
    
    df_imp = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False)
    
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_imp, x="importance", y="feature")
    plt.title(f"{name} - Feature Importance (Gain)")
    plt.tight_layout()
    plt.show()
    
    print(df_imp)

In [ ]:
# Early Stopping Demo (outside GridSearchCV)

print("\U0001f539 Early Stopping Demo (Regression)...")

# Get best params from grid search (regression model)
reg_pipe = [p for n, p in pipelines if "Regressor" in n]
if reg_pipe:
    best_params = reg_pipe[0].named_steps["clf"].get_params()
    
    X_train_es, X_val, y_train_es, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=42
    )
    
    # Preprocess
    X_train_transformed = preprocessor.fit_transform(X_train_es)
    X_val_transformed = preprocessor.transform(X_val)
    
    xgb_es = XGBRegressor(
        n_estimators=1000,
        max_depth=best_params.get("max_depth", 5),
        learning_rate=best_params.get("learning_rate", 0.05),
        subsample=best_params.get("subsample", 0.8),
        colsample_bytree=best_params.get("colsample_bytree", 0.8),
        random_state=42,
        verbosity=0,
        early_stopping_rounds=50,
        eval_metric="rmse"
    )
    
    xgb_es.fit(
        X_train_transformed, y_train_es,
        eval_set=[(X_train_transformed, y_train_es), (X_val_transformed, y_val)],
        verbose=False
    )
    
    print(f"\u2705 Best iteration: {xgb_es.best_iteration}")
    print(f"   Best RMSE: {xgb_es.best_score:.4f}")

In [ ]:
# Learning Curves

if reg_pipe:
    eval_results = xgb_es.evals_result()
    
    plt.figure(figsize=(10, 5))
    plt.plot(eval_results["validation_0"]["rmse"], label="Train")
    plt.plot(eval_results["validation_1"]["rmse"], label="Validation")
    plt.axvline(xgb_es.best_iteration, color='r', linestyle='--', label=f"Best iteration: {xgb_es.best_iteration}")
    plt.xlabel("Boosting Round")
    plt.ylabel("RMSE")
    plt.title("XGBoost Learning Curves")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Regression Diagnostics

def regression_diagnostics(model, X, y, name="Model"):
    y_pred = model.predict(X)
    residuals = y - y_pred
    fitted = y_pred

    plt.figure(figsize=(12, 10))
    
    plt.subplot(2,2,1)
    sns.scatterplot(x=fitted, y=residuals)
    plt.axhline(0, color='r', linestyle='--')
    plt.xlabel("Fitted values")
    plt.ylabel("Residuals")
    plt.title(f"{name} - Residuals vs Fitted")
    
    plt.subplot(2,2,2)
    sns.histplot(residuals, kde=True, bins=30)
    plt.title(f"{name} - Residual Distribution")
    
    plt.subplot(2,2,3)
    sns.scatterplot(x=fitted, y=np.sqrt(np.abs(residuals)))
    plt.xlabel("Fitted values")
    plt.ylabel("Sqrt(|Residuals|)")
    plt.title(f"{name} - Scale-Location")
    
    plt.subplot(2,2,4)
    sns.scatterplot(x=y, y=fitted)
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{name} - Predicted vs Actual")
    
    plt.tight_layout()
    plt.show()

for name, pipe in pipelines:
    if "Regressor" in name:
        regression_diagnostics(pipe, X_train_full, y_train_full, f"{name} (Train)")
        regression_diagnostics(pipe, X_test, y_test, f"{name} (Test)")

In [ ]:
# Classification Diagnostics

for name, pipe in pipelines:
    if "Classifier" in name:
        y_test_binary = (y_test > np.median(y_test)).astype(int)
        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        cm = confusion_matrix(y_test_binary, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
        axes[0].set_xlabel("Predicted")
        axes[0].set_ylabel("Actual")
        axes[0].set_title(f"{name} - Confusion Matrix")
        
        fpr, tpr, _ = roc_curve(y_test_binary, y_proba)
        auc = roc_auc_score(y_test_binary, y_proba)
        axes[1].plot(fpr, tpr, label=f"AUC = {auc:.4f}")
        axes[1].plot([0, 1], [0, 1], 'r--')
        axes[1].set_xlabel("False Positive Rate")
        axes[1].set_ylabel("True Positive Rate")
        axes[1].set_title(f"{name} - ROC Curve")
        axes[1].legend()
        
        prec, rec, _ = precision_recall_curve(y_test_binary, y_proba)
        axes[2].plot(rec, prec)
        axes[2].set_xlabel("Recall")
        axes[2].set_ylabel("Precision")
        axes[2].set_title(f"{name} - Precision-Recall Curve")
        
        plt.tight_layout()
        plt.show()
        
        print(classification_report(y_test_binary, y_pred))

In [ ]:
# Profile Plots

def plot_profiles(best_pipeline, X, y_true):
    y_pred = best_pipeline.predict(X)
    
    n_cols = 3
    n_features = X.shape[1]
    n_rows = math.ceil(n_features / n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = axes.flatten()  

    for i, col in enumerate(X.columns):
        temp = pd.DataFrame({
            col: X[col],
            "y_true": y_true,
            "y_pred": y_pred
        })

        grouped = temp.groupby(col).agg(
            count=("y_true", "size"),
            avg_true=("y_true", "mean"),
            avg_pred=("y_pred", "mean")
        ).reset_index().sort_values(col)

        ax1 = axes[i]
        ax1.bar(grouped[col], grouped["count"], alpha=0.3)
        ax1.set_xlabel(col)
        ax1.set_ylabel("Population")

        ax2 = ax1.twinx()
        ax2.plot(grouped[col], grouped["avg_true"], marker="o", label="Actual Target")
        ax2.plot(grouped[col], grouped["avg_pred"], marker="o", linestyle="--", label="Predicted Target")
        ax2.set_ylabel("Target Value")

        ax1.set_title(col)
        ax2.legend(loc="upper right")

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

plot_profiles(best_model_pipeline, X_train_full, y_train_full)

In [ ]:
# future work:
#   SHAP values for detailed feature interaction analysis (pip install shap)
#   RandomizedSearchCV or Optuna for faster hyperparameter search on large grids
#   GPU acceleration: tree_method="gpu_hist" for faster training
#   monotone_constraints for business-logic feature constraints
#   custom objective functions for specialized loss